# Experi v3: governed A/B test analysis on Databricks

**What this notebook does, end to end**
1. Simulates raw product events (visits, purchases) and writes them to a Unity Catalog Delta table.
2. Records the table's Delta **version at experiment launch**, so pre-experiment covariates are read from a frozen snapshot (time travel).
3. Computes per-user covariates and outcomes in **Spark**.
4. Applies **CUPED** variance reduction.
5. Runs **Bayesian inference**: Beta-Binomial on raw conversion, and a Normal posterior on the CUPED-adjusted outcome (CUPED values are no longer 0/1, so a Beta model doesn't apply to them).
6. Logs parameters, Delta versions and metrics to **MLflow**.
7. Writes per-user metrics and the final result to Unity Catalog tables, so lineage runs `raw_events → user_metrics → experiment_results`.

All data is simulated. The true effect is known (`TRUE_RELATIVE_LIFT`), so we can check the pipeline recovers it.

In [0]:
import numpy as np
import pandas as pd
from datetime import date, timedelta
from scipy import stats
import mlflow
from pyspark.sql import functions as F

# ---- Config ----
CATALOG = "workspace"          # default catalog in Databricks Free Edition; change if yours differs
SCHEMA = "experi"
EXP_NAME = "checkout_cta_test"
EXP_START = date(2026, 5, 1)
EXP_END = date(2026, 5, 14)    # inclusive
PRE_DAYS = 30                  # covariate window: the 30 days before launch
N_USERS = 20_000
TRUE_RELATIVE_LIFT = 0.10      # treatment raises purchase probability per visit by 10%
SEED = 42

RAW = f"{CATALOG}.{SCHEMA}.raw_events"
ASSIGN = f"{CATALOG}.{SCHEMA}.assignments"
USER_METRICS = f"{CATALOG}.{SCHEMA}.user_metrics"
RESULTS = f"{CATALOG}.{SCHEMA}.experiment_results"

PRE_START = EXP_START - timedelta(days=PRE_DAYS)
PRE_END = EXP_START - timedelta(days=1)

In [0]:
# Fresh start each run, so Delta versions are predictable (0 = pre-period load).
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
for t in [RAW, ASSIGN, USER_METRICS]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
# RESULTS is kept on purpose: it's an append-only history of every analysis run.

def latest_version(table):
    return spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").collect()[0]["version"]

def read_at_version(table, version):
    return spark.sql(f"SELECT * FROM {table} VERSION AS OF {version}")

## 1. Simulate users and raw events

Each user has their own visit rate and purchase-per-visit rate, so past behavior predicts future behavior. That correlation is what CUPED uses.

In [0]:
rng = np.random.default_rng(SEED)
users = pd.DataFrame({
    "user_id": np.arange(N_USERS),
    "visit_rate": rng.beta(2, 8, N_USERS),      # chance of visiting on a given day (~20% avg)
    "purchase_rate": rng.beta(2, 20, N_USERS),  # chance of buying, given a visit (~9% avg)
    "variant": np.where(rng.random(N_USERS) < 0.5, "treatment", "control"),
})

def simulate_events(users, start, end, lift_for_treatment, rng):
    days = pd.date_range(start, end, freq="D").date
    n, d = len(users), len(days)
    visited = rng.random((n, d)) < users["visit_rate"].values[:, None]
    p = users["purchase_rate"].values.copy()
    p[users["variant"].values == "treatment"] *= (1 + lift_for_treatment)
    purchased = visited & (rng.random((n, d)) < p[:, None])

    rows = []
    for kind, mask in [("visit", visited), ("purchase", purchased)]:
        ui, di = np.nonzero(mask)
        rows.append(pd.DataFrame({
            "user_id": users["user_id"].values[ui],
            "event_date": np.array(days)[di],
            "event_type": kind,
            "revenue": np.round(rng.lognormal(3.5, 0.6, len(ui)), 2) if kind == "purchase" else 0.0,
        }))
    return pd.concat(rows, ignore_index=True)

pre_events = simulate_events(users, PRE_START, PRE_END, 0.0, rng)               # no treatment before launch
exp_events = simulate_events(users, EXP_START, EXP_END, TRUE_RELATIVE_LIFT, rng)
print(f"pre-period events: {len(pre_events):,} | experiment-period events: {len(exp_events):,}")

pre-period events: 130,434 | experiment-period events: 61,424


## 2. Write to Delta and pin the launch snapshot

- **Version 0**: pre-period events (what existed at launch). We save this version number.
- **Version 1**: experiment-period events arrive.
- **Version 2**: a late backfill of pre-period purchases lands after launch (common in real pipelines).

Reading covariates `VERSION AS OF` the launch snapshot means the covariate window can't be changed by later writes, so re-running the analysis months later gives the same answer.

In [0]:
spark.createDataFrame(pre_events).write.format("delta").saveAsTable(RAW)
LAUNCH_VERSION = latest_version(RAW)

spark.createDataFrame(users[["user_id", "variant"]]) \
    .withColumn("assigned_at", F.lit(EXP_START)) \
    .write.format("delta").saveAsTable(ASSIGN)

spark.createDataFrame(exp_events).write.format("delta").mode("append").saveAsTable(RAW)

# Late backfill: ~3% extra pre-period purchases recorded after launch
late = pre_events[pre_events.event_type == "purchase"].sample(frac=0.03, random_state=SEED)
spark.createDataFrame(late).write.format("delta").mode("append").saveAsTable(RAW)

ANALYSIS_VERSION = latest_version(RAW)
print(f"launch snapshot = v{LAUNCH_VERSION}, analysis snapshot = v{ANALYSIS_VERSION}")
display(spark.sql(f"DESCRIBE HISTORY {RAW}").select("version", "timestamp", "operation"))

launch snapshot = v0, analysis snapshot = v2


version,timestamp,operation
2,2026-10-23T21:37:33.000Z,WRITE
1,2026-10-23T21:37:30.000Z,WRITE
0,2026-10-23T21:37:16.000Z,CREATE TABLE AS SELECT


## 3. Covariates and outcomes in Spark

- **Covariate** (`pre_purchases`): purchases in the 30 days before launch, read from the **launch snapshot**.
- **Outcome** (`converted`): 1 if the user purchased at least once during the experiment, read from the analysis snapshot.

In [0]:
pre = (read_at_version(RAW, LAUNCH_VERSION)
       .filter(F.col("event_date").between(F.lit(PRE_START), F.lit(PRE_END)))
       .groupBy("user_id")
       .agg(F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("pre_purchases"),
            F.sum(F.when(F.col("event_type") == "visit", 1).otherwise(0)).alias("pre_visits")))

post = (read_at_version(RAW, ANALYSIS_VERSION)
        .filter(F.col("event_date").between(F.lit(EXP_START), F.lit(EXP_END)))
        .groupBy("user_id")
        .agg(F.max(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("converted"),
             F.sum("revenue").alias("revenue")))

user_metrics = (spark.table(ASSIGN)
                .join(pre, "user_id", "left")
                .join(post, "user_id", "left")
                .fillna(0, subset=["pre_purchases", "pre_visits", "converted", "revenue"]))

# Why pinning matters: the backfill changes the covariate if you read the latest version instead.
pre_latest_total = (spark.table(RAW)
                    .filter(F.col("event_date").between(F.lit(PRE_START), F.lit(PRE_END)) & (F.col("event_type") == "purchase"))
                    .count())
pre_pinned_total = user_metrics.agg(F.sum("pre_purchases")).collect()[0][0]
print(f"pre-period purchases at launch snapshot: {pre_pinned_total:,} | at latest version: {pre_latest_total:,}")

pre-period purchases at launch snapshot: 10,724 | at latest version: 11,046


## 4. CUPED

`y_cuped = y − θ · (x − mean(x))`, with `θ = cov(x, y) / var(x)`. This removes the part of each user's outcome that was predictable from their pre-period behavior. The mean doesn't change; the variance drops.

In [0]:
s = user_metrics.agg(F.covar_samp("converted", "pre_purchases").alias("cov_xy"),
                     F.var_samp("pre_purchases").alias("var_x"),
                     F.avg("pre_purchases").alias("mean_x"),
                     F.corr("converted", "pre_purchases").alias("rho")).collect()[0]
theta = s["cov_xy"] / s["var_x"]

user_metrics = user_metrics.withColumn(
    "converted_cuped", F.col("converted") - F.lit(theta) * (F.col("pre_purchases") - F.lit(s["mean_x"])))
user_metrics.write.format("delta").mode("overwrite").saveAsTable(USER_METRICS)

v = spark.table(USER_METRICS).agg(F.var_samp("converted").alias("raw"),
                                   F.var_samp("converted_cuped").alias("adj")).collect()[0]
var_reduction = 1 - v["adj"] / v["raw"]
print(f"theta = {theta:.4f} | corr(x, y) = {s['rho']:.3f} | variance reduction = {var_reduction:.1%}")
print(f"(theory: variance reduction = corr^2 = {s['rho']**2:.1%})")

theta = 0.1081 | corr(x, y) = 0.235 | variance reduction = 5.5%
(theory: variance reduction = corr^2 = 5.5%)


## 5. Bayesian inference

- **Raw conversion (0/1)**: Beta(1,1) prior → Beta posterior per arm. Valid because the outcome is binary.
- **CUPED-adjusted outcome**: no longer 0/1, so a Beta model would be wrong. With a flat prior and 20K users, each arm's mean has a Normal posterior: `N(mean, var/n)`. The difference of two Normals is Normal, so P(treatment > control) has a closed form.

Decision rule: **SHIP** if P(treatment > control) > 0.95, otherwise **HOLD**.

In [0]:
arm = {r["variant"]: r for r in spark.table(USER_METRICS).groupBy("variant").agg(
    F.count("*").alias("n"), F.sum("converted").alias("conversions"),
    F.avg("converted").alias("mean_raw"), F.var_samp("converted").alias("var_raw"),
    F.avg("converted_cuped").alias("mean_adj"), F.var_samp("converted_cuped").alias("var_adj")).collect()}
c, t = arm["control"], arm["treatment"]

# (a) Beta-Binomial on raw conversion
draws = 200_000
bc = rng.beta(1 + c["conversions"], 1 + c["n"] - c["conversions"], draws)
bt = rng.beta(1 + t["conversions"], 1 + t["n"] - t["conversions"], draws)
p_raw = (bt > bc).mean()
ci_raw = np.percentile(bt - bc, [2.5, 97.5])

# (b) Normal posterior on CUPED-adjusted outcome
diff = t["mean_adj"] - c["mean_adj"]
sd = np.sqrt(t["var_adj"] / t["n"] + c["var_adj"] / c["n"])
p_cuped = stats.norm.cdf(diff / sd)
ci_cuped = (diff - 1.96 * sd, diff + 1.96 * sd)
rel_lift = diff / c["mean_raw"]
decision = "SHIP" if p_cuped > 0.95 else "HOLD"

print(f"Raw     : P(T>C) = {p_raw:.3f} | 95% CrI = [{ci_raw[0]:.4f}, {ci_raw[1]:.4f}]  width {ci_raw[1]-ci_raw[0]:.4f}")
print(f"CUPED   : P(T>C) = {p_cuped:.3f} | 95% CrI = [{ci_cuped[0]:.4f}, {ci_cuped[1]:.4f}]  width {ci_cuped[1]-ci_cuped[0]:.4f}")
print(f"Lift    : {diff:.4f} absolute ({rel_lift:.1%} relative) | control conversion {c['mean_raw']:.3f}")
print(f"Decision: {decision}")

Raw     : P(T>C) = 1.000 | 95% CrI = [0.0146, 0.0374]  width 0.0228
CUPED   : P(T>C) = 1.000 | 95% CrI = [0.0132, 0.0353]  width 0.0222
Lift    : 0.0242 absolute (12.0% relative) | control conversion 0.203
Decision: SHIP


## 6. Log to MLflow

Parameters include both Delta versions, so anyone can re-run this exact analysis on the exact same data. The raw table is also logged as an MLflow dataset input pinned to the analysis version.

In [0]:
user = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{user}/experi")

with mlflow.start_run(run_name=EXP_NAME) as run:
    mlflow.log_params({
        "experiment": EXP_NAME, "exp_start": str(EXP_START), "exp_end": str(EXP_END),
        "metric": "converted", "covariate": "pre_purchases", "covariate_window_days": PRE_DAYS,
        "source_table": RAW, "launch_version": LAUNCH_VERSION, "analysis_version": ANALYSIS_VERSION,
        "inference": "Normal posterior on CUPED-adjusted outcome; Beta-Binomial on raw",
        "decision_rule": "SHIP if P(T>C) > 0.95", "n_users": c["n"] + t["n"],
    })
    mlflow.log_metrics({
        "cuped_theta": theta, "variance_reduction": var_reduction,
        "prob_treatment_better": p_cuped, "prob_treatment_better_raw": p_raw,
        "abs_lift": diff, "rel_lift": rel_lift,
        "ci_lower": ci_cuped[0], "ci_upper": ci_cuped[1],
    })
    try:
        mlflow.log_input(mlflow.data.load_delta(table_name=RAW, version=ANALYSIS_VERSION), context="raw_events")
    except Exception as e:
        print(f"(dataset logging skipped: {e})")
    RUN_ID = run.info.run_id
print(f"MLflow run: {RUN_ID}")

MLflow run: 43dfde5aa46a4e7c92f48e3ee2ad54cd


## 7. Write the result to Unity Catalog

The result row is computed from `user_metrics`, so Unity Catalog's lineage graph shows `raw_events → user_metrics → experiment_results`. The row also stores the MLflow run ID and both Delta versions.

To see lineage: Catalog → `experi` → `experiment_results` → **Lineage** tab.

In [0]:
result = (spark.table(USER_METRICS)
          .agg(F.count("*").alias("n_users"))
          .select(F.lit(EXP_NAME).alias("experiment"),
                  F.lit(RUN_ID).alias("mlflow_run_id"),
                  F.lit(RAW).alias("source_table"),
                  F.lit(int(LAUNCH_VERSION)).alias("launch_version"),
                  F.lit(int(ANALYSIS_VERSION)).alias("analysis_version"),
                  "n_users",
                  F.lit(float(var_reduction)).alias("variance_reduction"),
                  F.lit(float(p_cuped)).alias("prob_treatment_better"),
                  F.lit(float(diff)).alias("abs_lift"),
                  F.lit(float(rel_lift)).alias("rel_lift"),
                  F.lit(decision).alias("decision"),
                  F.current_timestamp().alias("analyzed_at")))
result.write.format("delta").mode("append").saveAsTable(RESULTS)
display(spark.table(RESULTS).orderBy(F.desc("analyzed_at")))

experiment,mlflow_run_id,source_table,launch_version,analysis_version,n_users,variance_reduction,prob_treatment_better,abs_lift,rel_lift,decision,analyzed_at
checkout_cta_test,43dfde5aa46a4e7c92f48e3ee2ad54cd,workspace.experi.raw_events,0,2,20000,0.05526600872914755,0.9999909760490732,0.024229791454144334,0.11951197185702886,SHIP,2026-10-23T21:38:04.394Z
